# TRANSFER LEARNING IN PYTORCH LIGHTNING


In [1]:
!pip install pytorch_lightning

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import pytorch_lightning as pl
import torchvision.models as models

import numpy as np
import torchmetrics
from torchmetrics import Metric



In [3]:
num_epochs = 15
batch_size = 8
learning_rate = 0.001

In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,0.5, 0.5),(0.5,0.5,0.5))]
)

In [5]:
train_dataset= torchvision.datasets.CIFAR10(root='./data', train = True, download = True, transform= transform)
test_dataset= torchvision.datasets.CIFAR10(root='./data', train = False, download = True, transform= transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, shuffle = True)

Files already downloaded and verified
Files already downloaded and verified


In [6]:



classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [7]:
num_classes= len(classes)

In [8]:

#loss_fn = nn.CrossEntropyLoss()

In [9]:
model = torchvision.models.vgg16()

In [10]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [11]:
import torch.nn as nn
from torchvision.models import vgg16,VGG16_Weights

class VGG16FineTune(nn.Module):
    def __init__(self, num_classes):
        super(VGG16FineTune, self).__init__()
        self.vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        # Freeze the layers except the last fully connected layers
        for param in self.vgg.parameters():
            param.requires_grad = False
        # Replace the last layer_
        self.vgg.classifier[6] = nn.Sequential(
             nn.Linear(4096, 1024),
             nn.ReLU(inplace = True),
             nn.Linear(1024, 512),
             nn.ReLU(inplace = True),
             nn.Linear(512, num_classes)
            
        )
            
             

    def forward(self, x):
        return self.vgg(x)





In [12]:
from torchmetrics import Metric

In [13]:
class VGGnet(pl.LightningModule):
    def __init__(self, num_classes):
        super().__init__()
        self.model = VGG16FineTune(num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task= "multiclass", num_classes= num_classes)
        self.f1_score = torchmetrics.F1Score(task="multiclass", num_classes= num_classes)



    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)
        accuracy = self.accuracy(scores,y)
        f1_score = self.f1_score(scores,y)
        self.log_dict({'train_loss': loss, 'train_accuracy': accuracy, 'train_f1_score':f1_score}, on_step = False, on_epoch= True, prog_bar= True)
        return {'loss':loss, 'scores':scores, "y":y}


    def validation_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)

    def test_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)

    def common_step(self,batch,batch_idx):

        x, y = batch
        scores = self.forward(x)
        loss = self.criterion(scores,y)
        return loss, scores, y

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.vgg.classifier[6].parameters(), lr=1e-3)
        return optimizer







In [14]:
import sys

In [15]:

from pytorch_lightning import Trainer

# Initialize the model
model = VGGnet(num_classes)

# Initialize the trainer
trainer = Trainer(max_epochs=20)  # Use gpus=1 if you have a GPU

# Train the model
trainer.fit(model, train_loader)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\user\Documents\project\new\ml-internship\week3\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
c:\Users\user\Documents\project\new\ml-internship\week3\.conda\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
-----

Epoch 11:  19%|█▉        | 1191/6250 [00:24<01:42, 49.45it/s, v_num=1, train_loss=1.010, train_accuracy=0.660, train_f1_score=0.660]

c:\Users\user\Documents\project\new\ml-internship\week3\.conda\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


### CHATGPT CODE

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

class VGG16FineTune(nn.Module):
    def __init__(self, num_classes):
        super(VGG16FineTune, self).__init__()
        # Load the pre-trained VGG16 model with the correct weights
        self.vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        # Freeze the layers except the last fully connected layers
        for param in self.vgg.parameters():
            param.requires_grad = False
        # Replace the last layer
        self.vgg.classifier[6] = nn.Sequential(
            nn.Linear()
        )

    def forward(self, x):
        return self.vgg(x)

import pytorch_lightning as pl
import torch.nn.functional as F

class LitVGG16(pl.LightningModule):
    def __init__(self, num_classes):
        super(LitVGG16, self).__init__()
        self.model = VGG16FineTune(num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        val_loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('val_loss', val_loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.vgg.classifier[6].parameters(), lr=1e-3)
        return optimizer

from pytorch_lightning import Trainer

# Initialize the model
model = LitVGG16(num_classes=10)

# Initialize the trainer
trainer = Trainer(max_epochs=10)  # Use gpus=1 if you have a GPU

# Train the model
trainer.fit(model, train_loader, val_loader)


In [ ]:
backbone

In [ ]:
layers

In [ ]:
model = torchvision.models.vgg16(pretrained = "True")

In [ ]:
model

In [ ]:
model = vggTransferLearning()

In [ ]:
model.avgpool = Identity()
model.classifer = nn.Linear(512,10)

In [ ]:
model

In [ ]:
trainer = pl.Trainer(max_epochs=5)

trainer.fit(model, train_loader)